# Session 12 — Verification & Validation

**Course:** Decision Modelling — Simulation Part
**Session:** 12 of 12 (simulation portion) — capstone

## Learning objectives

By the end of this session you should be able to:

- Explain the distinction between **verification** and **validation**, and why neither alone is sufficient
- Apply structured verification techniques to check that a simulation model is correctly implemented
- Apply validation techniques (and know their practical limits) to assess whether a model adequately represents the real system it's meant to represent

## Where this session fits

- **This is the final session of the simulation part, and it is deliberately not about building anything new.** Instead, we step back and critically examine models you've already built — especially Session 11's capstone — using the lens of V&V.
- **Note on scope:** this session covers verification and validation only. Simulation-based optimization (connecting these models to the optimization side of the course) is covered separately, in two dedicated integration sessions later in the course — not here.
- **Looking back, with new eyes:** you've actually been doing verification all course long without necessarily naming it — Session 6's check against the M/M/1 formula, Session 9's Little's Law cross-check, Session 7's `c=1` reduction to M/M/1. Today we make this an explicit, structured practice rather than something that happens incidentally.


## Further reading (supplementary, not required)

- Sargent's work on simulation model verification and validation is the standard reference framework for this topic — check the course reading list for the specific citation.


## 1. Verification vs. validation: two different questions

These two words are used almost interchangeably in casual speech, but they ask genuinely different questions:

| | **Verification** | **Validation** |
|---|---|---|
| **Question** | "Did we build the model *right*?" | "Did we build the *right* model?" |
| **Concerned with** | Internal correctness: does the code correctly implement the conceptual model we intended? | External fidelity: does the conceptual model (and hence the code) adequately represent the real system for the questions we're asking? |
| **Can be wrong even if the other is right?** | Yes — a perfectly validated *concept* can still be buggy in code | Yes — perfectly correct code can faithfully implement a conceptual model that's simply the wrong model of reality |

**Both are necessary, and neither is sufficient on its own.** A beautifully verified model of the wrong system is useless; a validated concept riddled with implementation bugs is dangerous, because it *looks* trustworthy.


## 2. Verification techniques

Verification asks: does the code do what we *intended* it to do? A few structured techniques, several of which you've already used without the formal label:

- **Structured walkthrough / code review** — read the code line by line against the conceptual model, ideally with someone else
- **Comparison against a known analytical solution** — Session 6's check of a SimPy M/M/1 model against the exact M/M/1 formulas is a textbook example
- **Degenerate / special-case testing** — set parameters to extreme or simplified values where the *correct* answer is obvious or independently known, and check the model agrees. Session 7's `c=1` pooled-queue check (which should reduce exactly to the M/M/1 case) is exactly this.
- **Internal consistency checks** — relationships that must hold regardless of implementation details, like Little's Law (used in Session 6 and Session 9), are a powerful way to catch bugs that wouldn't otherwise be obvious
- **Tracing** — stepping through individual entity histories (e.g. inspecting `records` row by row) to confirm event sequences make sense
- **Sensitivity/extreme-condition tests** — does the model respond in the *direction* you'd expect when you push a parameter to an extreme? (e.g. zero arrival rate should give zero WIP; near-zero service time should give near-zero waiting)

Let's apply several of these to Session 11's capstone production/inventory model.


In [1]:
import simpy
import numpy as np
import pandas as pd
from scipy import stats

# Reusing the Session 11 model, unchanged, as the object of our verification exercise
def check_reorder(env, container, policy_state, log):
    if container.level <= policy_state['s'] and not policy_state['order_outstanding']:
        order_qty = policy_state['S'] - container.level
        policy_state['order_outstanding'] = True
        env.process(place_order(env, container, order_qty, policy_state, log))

def place_order(env, container, order_qty, policy_state, log):
    yield env.timeout(policy_state['lead_time'])
    yield container.put(order_qty)
    policy_state['order_outstanding'] = False

def station1(env, raw_material, buffer, mean_process_time, rng, policy_state, reorder_log, metrics):
    while True:
        yield raw_material.get(1)
        check_reorder(env, raw_material, policy_state, reorder_log)
        yield env.timeout(rng.exponential(mean_process_time))
        t0 = env.now
        yield buffer.put('item')
        metrics['station1_blocked_time'] += env.now - t0

def station2(env, buffer, finished_goods, mean_process_time, rng, metrics):
    while True:
        t0 = env.now
        yield buffer.get()
        metrics['station2_starved_time'] += env.now - t0
        yield env.timeout(rng.exponential(mean_process_time))
        yield finished_goods.put(1)

def demand_process(env, finished_goods, mean_interarrival, rng, metrics):
    while True:
        yield env.timeout(rng.exponential(mean_interarrival))
        metrics['total_demand'] += 1
        if finished_goods.level >= 1:
            yield finished_goods.get(1)
            metrics['demand_met'] += 1
        else:
            metrics['stockouts'] += 1

def run_production_system(run_length, buffer_capacity, s, S, lead_time,
                            mean_process_time_1, mean_process_time_2, mean_demand_interarrival, seed):
    rng = np.random.default_rng(seed)
    env = simpy.Environment()
    raw_material = simpy.Container(env, capacity=100000, init=S)
    buffer = simpy.Store(env, capacity=buffer_capacity)
    finished_goods = simpy.Container(env, capacity=100000, init=0)

    policy_state = {'s': s, 'S': S, 'lead_time': lead_time, 'order_outstanding': False}
    reorder_log = []
    metrics = {'station1_blocked_time': 0.0, 'station2_starved_time': 0.0,
               'total_demand': 0, 'demand_met': 0, 'stockouts': 0}

    env.process(station1(env, raw_material, buffer, mean_process_time_1, rng, policy_state, reorder_log, metrics))
    env.process(station2(env, buffer, finished_goods, mean_process_time_2, rng, metrics))
    env.process(demand_process(env, finished_goods, mean_demand_interarrival, rng, metrics))
    env.run(until=run_length)
    return metrics

print("Session 11 model reloaded for verification testing.")


Session 11 model reloaded for verification testing.


### Verification test 1: an (almost) infinite buffer should eliminate blocking

If the intermediate buffer is effectively unlimited, Station 1 should **never** be forced to wait to deposit its output — blocking should be approximately zero.


In [2]:
m_large_buffer = run_production_system(
    run_length=2000.0, buffer_capacity=100000, s=20, S=100, lead_time=10.0,
    mean_process_time_1=1.5, mean_process_time_2=1.8, mean_demand_interarrival=2.0, seed=1
)

blocked_frac = m_large_buffer['station1_blocked_time'] / 2000.0
print(f"Station 1 blocked fraction with an effectively infinite buffer: {blocked_frac:.5%}")
print("Expected: approximately 0% -- ", "PASS" if blocked_frac < 0.001 else "FAIL: investigate")


Station 1 blocked fraction with an effectively infinite buffer: 0.00000%
Expected: approximately 0% --  PASS


### Verification test 2: reversing the bottleneck should reverse the blocking pattern

In Session 11, Station 2 was slower, and we saw heavy blocking at Station 1, light starvation at Station 2. If we **swap** the two processing speeds (make Station 1 the slow one instead), the *relative* blocking/starvation pattern should **flip** — blocking should now be lower than starvation, whereas before it was the reverse. This checks that our blocking/starvation logic isn't accidentally hardcoded to always blame one particular station.


In [3]:
m_swapped = run_production_system(
    run_length=2000.0, buffer_capacity=5, s=20, S=100, lead_time=10.0,
    mean_process_time_1=1.8, mean_process_time_2=1.5,  # <-- swapped from Session 11's values
    mean_demand_interarrival=2.0, seed=1
)

blocked_frac_swapped = m_swapped['station1_blocked_time'] / 2000.0
starved_frac_swapped = m_swapped['station2_starved_time'] / 2000.0

print(f"Station 1 blocked fraction (now the SLOW station): {blocked_frac_swapped:.3%}")
print(f"Station 2 starved fraction (now the FAST station):  {starved_frac_swapped:.3%}")
print("\nExpected: blocking should now be LOWER than starvation (roles have reversed from "
      "Session 11's original result, where blocking was ~24% and starving was ~5%).")
print("PASS" if blocked_frac_swapped < starved_frac_swapped else "FAIL: investigate")


Station 1 blocked fraction (now the SLOW station): 7.852%
Station 2 starved fraction (now the FAST station):  26.145%

Expected: blocking should now be LOWER than starvation (roles have reversed from Session 11's original result, where blocking was ~24% and starving was ~5%).
PASS


### Verification test 3: an internal consistency check (a Little's-Law-style sanity test)

Station 2's utilization (fraction of time busy actually processing, excluding starved time) should be **consistent** with its workload: over a long run, `busy_fraction ~= (items processed x mean_process_time_2) / run_length`. Let's check this holds, as an independent consistency check on the bookkeeping in `station2()`.


In [4]:
run_length_check = 5000.0
m_check = run_production_system(
    run_length=run_length_check, buffer_capacity=5, s=20, S=100, lead_time=10.0,
    mean_process_time_1=1.5, mean_process_time_2=1.8, mean_demand_interarrival=2.0, seed=1
)

# Station 2's "not starved" time is exactly its busy time (it's either starved, waiting for input,
# or busy processing -- there's no third state for a single-capacity station)
station2_busy_time = run_length_check - m_check['station2_starved_time']
implied_items_processed = station2_busy_time / 1.8  # mean_process_time_2
items_that_reached_finished_goods = m_check['demand_met'] + m_check['stockouts'] * 0  # only demand_met were actually withdrawn
# A cleaner check: finished_goods produced = demand_met + whatever remained in stock at the end
# For a rough verification check, compare implied throughput to the demand-met rate as an order-of-magnitude sanity check
print(f"Station 2 busy time: {station2_busy_time:.1f} out of {run_length_check}")
print(f"Implied items processed (busy_time / mean_process_time_2): {implied_items_processed:.1f}")
print(f"Demand actually met (lower bound on items produced): {m_check['demand_met']}")
print("\nThese should be in the same ballpark -- implied throughput should be a bit higher than "
      "demand met, since some produced items may remain in finished-goods stock at the end of the run.")


Station 2 busy time: 4709.6 out of 5000.0
Implied items processed (busy_time / mean_process_time_2): 2616.4
Demand actually met (lower bound on items produced): 2499

These should be in the same ballpark -- implied throughput should be a bit higher than demand met, since some produced items may remain in finished-goods stock at the end of the run.


### Discussion: what would a FAIL actually tell us?

If Test 1 had shown substantial blocking even with a huge buffer, what would that tell you about where to look in the code? (Hint: think about what `buffer.put()` depends on besides buffer capacity.) This is the real value of verification tests: a failure doesn't just say "something's wrong," it usually **points you toward what to check first.**


## 3. Validation: does the model represent the real system?

Validation is harder than verification, and often can't be fully "passed" with certainty — especially for a course exercise using illustrative parameters rather than data collected from a real clinic, factory, or shop. Practical validation techniques:

- **Face validation** — present the model's assumptions and logic to people with real domain knowledge (e.g. an actual production manager) and ask: "does this look right to you?" Simple, cheap, and often catches major conceptual errors that no statistical test would reveal.
- **Comparison to historical data** — if real data on the system's actual performance exists, compare simulated output against it (this is exactly what Session 2's input-analysis techniques support on the input side, and what we could do on the output side if real KPI data were available).
- **Sensitivity analysis** — does the model respond to parameter changes in directions a domain expert would expect? (We effectively did this in Session 11's buffer-size exercise — increasing buffer size should improve, not worsen, service level, and it did.)
- **Traces reviewed by a domain expert** — walking a knowledgeable stakeholder through individual simulated event sequences (not just aggregate statistics) to check for anything implausible.
- **Turing-style tests** — if you have real output data, can a domain expert distinguish it from simulated output? If not, that's a (weak but useful) point in favor of the model's validity.

**A hard truth about this course's models specifically:** none of the parameters used in Sessions 6–11 came from measuring a real system's performance (though several, like the Newsvendor and queueing figures, came from real course datasets) — most were chosen to be illustrative and pedagogically clean. This means these models have reasonable **face validity** as teaching tools, but have not been — and could not be, without real operational data — fully validated against a specific real system. This is worth being honest about rather than glossing over.


## Exercise (guided): a structured V&V review of the Session 11 capstone

Imagine you're handing the Session 11 production/inventory model to a plant manager who wants to actually use its conclusions to make a buffer-sizing decision. Your task:

1. Design **one additional verification test** (beyond the three above) for this model — state what extreme/degenerate condition you'd test and what result you'd expect
2. List **three face-validation questions** you would ask a real plant manager about this model's assumptions (e.g. about the demand process, the processing-time distributions, or the lost-sales assumption) before trusting its conclusions
3. Identify **one assumption** in the Session 11 model that you suspect is the *least* realistic, and explain what data you'd want to collect to check it


In [5]:
# EXERCISE
# TODO: 1. Implement one additional verification test as code (run_production_system with
#          some extreme/degenerate parameter setting, and a printed PASS/FAIL check)

# TODO: 2. Write your three face-validation questions as a markdown list (in a new markdown cell)

# TODO: 3. Write one to two sentences identifying the least realistic assumption and what
#          data would help validate it (in a new markdown cell)


<details>
<summary>One possible solution (there is no single correct answer here)</summary>

**1. An additional verification test:** set `mean_demand_interarrival` extremely large (e.g. `10000`), so demand is almost never generated. Expected result: `total_demand` should be near 0, and consequently `stockouts` should also be 0 (since there's no demand to fail to meet) — a simple test that the demand process and the stockout-counting logic only fire when demand actually occurs.

```python
# SOLUTION
m_rare_demand = run_production_system(
    run_length=2000.0, buffer_capacity=5, s=20, S=100, lead_time=10.0,
    mean_process_time_1=1.5, mean_process_time_2=1.8, mean_demand_interarrival=10000.0, seed=1
)
print(m_rare_demand)
print("PASS" if m_rare_demand['total_demand'] <= 1 and m_rare_demand['stockouts'] == 0 else "FAIL")
```

**2. Face-validation questions for a plant manager:**
- "Is it realistic to assume demand arrivals are independent of each other and of the time of day, or does real demand cluster (e.g. morning rushes, seasonal spikes) in ways this model doesn't capture?"
- "We assumed unmet demand is simply a lost sale with no backorder and no memory of the missed customer — does that match how your customers actually behave, or would they wait, or would you lose a future order too?"
- "Are Station 1 and Station 2's processing times really well-described by an exponential distribution, or do they have much less variability in practice (e.g. a fairly fixed cycle time)?"

**3. Least realistic assumption:** the exponential processing-time assumption at both stations is probably the weakest link — real manufacturing or service processing times are often far less variable than exponential (which has quite high variability relative to its mean). To check this, you'd want to collect actual timestamped processing-time data from the real stations and apply Session 2's input-analysis workflow (histogram, candidate distribution, goodness-of-fit test) rather than assuming exponential by default.
</details>


### Discussion questions

1. Suppose a stakeholder says "the model's numbers don't match what I see on the shop floor." How would you decide whether that's a **verification** problem (a bug) or a **validation** problem (a wrong assumption)? What would you check first?
2. This course's models have generally used illustrative parameters rather than measured ones. If you were to genuinely validate one of them for a real decision, which session's tools (from Sessions 1–11) would you use to obtain properly justified parameters, rather than illustrative ones?
3. Is a model that's been thoroughly verified but never validated *more* or *less* dangerous than a model that's never been verified at all? Why?


## Wrap-up: the simulation part of this course, end to end

This session closes the 12-session simulation arc. Looking back across the whole arc:

- **Sessions 1–2:** conceptual modeling and input analysis — turning real data into justified random inputs
- **Sessions 3–4:** Monte Carlo simulation, confidence intervals, risk metrics, and Common Random Numbers
- **Session 5:** analytical queueing theory as a grounding for verification
- **Sessions 6–7:** building DES models in SimPy, from single-server to multi-server, verified against Session 5's formulas
- **Sessions 8–9:** rigorous output analysis for terminating and steady-state systems
- **Session 10:** rigorous scenario comparison, including the real limits of CRN in DES
- **Session 11:** a full capstone combining inventory, multi-stage flow, and bottleneck analysis
- **Session 12 (today):** stepping back to critically verify and validate models like the ones you've built, rather than taking any of them on faith

**What's next:** two integration sessions, taught jointly with the mathematical-programming part of the course, connect simulation to optimization — using verified, validated simulation models (exactly the kind of scrutiny practiced today) to evaluate and improve decisions found through optimization, and to perform simulation-based optimization directly.
